In [3]:
import pandas as pd
import numpy as np
import ast
import time
import re
import os
from tqdm import tqdm
from sentence_transformers import SentenceTransformer
from sklearn.cluster import MiniBatchKMeans
import torch


#### Data Processing

We've taken following steps to clean the data.
1. **Parsed String-Lists**: Columns like ingredients were just useless strings (e.g., '["item 1", "item 2"]'). We converted them into actual Python lists (e.g., ["item 1", "item 2"]) that a program can read and search. We did this for ingredients, steps, and nutrition.

2. **Expanded Nutrition Data:** We took the nutrition list (e.g., [250, 10.5, ...]) and split it into its own set of new, separate columns: calories, total_fat_g, sugar_g, sodium_mg, protein_g, etc. This makes nutritional analysis much easier.

3. **Removed "Useless" Recipes:** We dropped any recipes that were missing critical information, specifically any that had no name, no ingredients, or no steps.

4. **Standardized Data Types:** We made sure numeric columns (like minutes) were actually stored as numbers, not text, so we can perform calculations on them.

5. **Filled Missing Descriptions:** We replaced any missing description values (which were NaN) with an empty string ('') to prevent errors in the future.

6. **Removed Unneeded Columns:** We dropped contributor_id and submitted because they aren't needed for our app.


In [4]:
# --- Configuration ---
INPUT_FILE = './Data/RAW_recipes.csv'

# Define the directory and filename
OUTPUT_DIR = './Data/recipies_cleaned/'
OUTPUT_FILENAME = 'recipes_cleaned.parquet'
OUTPUT_FILE = os.path.join(OUTPUT_DIR, OUTPUT_FILENAME)

# Create the output directory if it doesn't exist
os.makedirs(OUTPUT_DIR, exist_ok=True)

# Columns that are stored as string-lists and need parsing
LIST_COLUMNS = ['nutrition', 'steps', 'ingredients']

# Columns for the nutrition list, in the order they appear
NUTRITION_COLUMNS = [
    'calories',
    'total_fat_g',
    'sugar_g',
    'sodium_mg',
    'protein_g',
    'saturated_fat_g',
    'carbohydrates_g'
]

In [5]:
# --- Helper Function ---
def safe_parse_list(s):
    try:
        return ast.literal_eval(s)
    except (ValueError, SyntaxError, TypeError):
        return []

# --- Main Processing Workflow ---
print("Starting data cleaning process...")
start_time = time.time()

# 1. Load Data
try:
    df = pd.read_csv(INPUT_FILE)
    print(f"Loaded {len(df)} recipes from {INPUT_FILE}")
except FileNotFoundError:
    print(f"ERROR: Input file not found at {INPUT_FILE}")
    exit()

# 2. Parse String-Literal Columns
print(f"Parsing string-literal columns: {LIST_COLUMNS}...")
for col in LIST_COLUMNS:
    df[col] = df[col].apply(safe_parse_list)

# 3. Handle Critical Missing Data (dropping recipies that are missing names, ingredients and steps)
df.dropna(subset=['name', 'ingredients', 'steps'], inplace=True)
df = df[df['ingredients'].str.len() > 0]
df = df[df['steps'].str.len() > 0]
print(f"Recipes after dropping critical missing data: {len(df)}")

# 4. Expand the 'nutrition' Column
print("Expanding 'nutrition' column into separate columns...")
nutrition_df = pd.DataFrame(
    df['nutrition'].tolist(),
    index=df.index,
    columns=NUTRITION_COLUMNS
)
for col in NUTRITION_COLUMNS:
    nutrition_df[col] = pd.to_numeric(nutrition_df[col], errors='coerce')

df = df.join(nutrition_df)
df.drop(columns=['nutrition'], inplace=True)

# 5. Filled Missing Descriptions:
print("Cleaning up other columns...")
df['description'] = df['description'].fillna('')

# Standardized Data Types for minutes, number of steps and ingredients column
df['minutes'] = pd.to_numeric(df['minutes'], errors='coerce')
df['n_steps'] = pd.to_numeric(df['n_steps'], errors='coerce')
df['n_ingredients'] = pd.to_numeric(df['n_ingredients'], errors='coerce')

# 6. Removed Unneeded Columns as they are not needed for analysis
df.drop(columns=['contributor_id', 'submitted'], inplace=True, errors='ignore')

# Saving the cleaned data
print("Saving cleaned data to Parquet file...")
df.to_parquet(OUTPUT_FILE, index=False)

end_time = time.time()
print(f"---")
print(f"Success! Cleaning complete in {end_time - start_time:.2f} seconds.")
print(f"Final dataset has {len(df)} recipes.")
print(f"Cleaned data saved to {OUTPUT_FILE}")

Starting data cleaning process...
Loaded 231637 recipes from ./Data/RAW_recipes.csv
Parsing string-literal columns: ['nutrition', 'steps', 'ingredients']...
Recipes after dropping critical missing data: 231635
Expanding 'nutrition' column into separate columns...
Cleaning up other columns...
Saving cleaned data to Parquet file...
---
Success! Cleaning complete in 9.73 seconds.
Final dataset has 231635 recipes.
Cleaned data saved to ./Data/recipies_cleaned/recipes_cleaned.parquet


In [9]:
if torch.backends.mps.is_available():
    device = torch.device("mps")  # Metal Performance Shaders GPU
    print("Using Apple MPS GPU")
else:
    device = torch.device("cpu")
    print("Using CPU")


Using Apple MPS GPU
